In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime
import urllib.request 

pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

# LA County Wildfire Data (January 2025)

## Data Sources

### Los Angeles County
See ArcGIS Viewer of Data Set here: 
https://data.lacounty.gov/datasets/6241d8e277a541a2b3645947d991c35e/explore?location=34.270980%2C-118.419280%2C8.35

Metadata: https://www.arcgis.com/sharing/rest/content/items/6241d8e277a541a2b3645947d991c35e/info/metadata/metadata.xml?format=default&output=html

### Ventura County

We contacted the Ventura County Department of Emergency Services directly, and received a shapefile and details for the sole evacuation alert issued during the study period. 

In [ ]:
url = 'https://services.arcgis.com/RmCCgQtiZLDCtblq/arcgis/rest/services/IPAWS_Jan_2025_Fire_Alerts/FeatureServer/2/query?outFields=*&where=1%3D1&f=geojson'
ipaws_raw = gpd.read_file('GEOJSON:' + url)
resp = urllib.request.urlretrieve(url, '../01_data/01_raw/la.geojson')


In [ ]:
ventura = gpd.read_file('../01_data/01_raw/oak_park_kenneth')

## Data Processing

### Los Angeles County

There are four times included with the data. We will filter to orders/warnings that were _in effect_ between January 7 and 10. We define _in effect_ here as after the "Effective" time and before the "Expires" time. In cases where the "Effective" time was not specified, it is assumed to be the time that the order was sent. 

In [ ]:
ipaws = ipaws_raw.copy()

# Format Time Columns
ipaws['Effective'] = pd.to_datetime(ipaws['Effective'], unit = 'ms', utc = True)
ipaws['Expires'] = pd.to_datetime(ipaws['Expires'], unit = 'ms', utc = True)
ipaws['CreateDate'] = pd.to_datetime(ipaws['CreateDate'], unit = 'ms', utc = True)
ipaws['Sent'] = pd.to_datetime(ipaws['Sent'], unit = 'ms', utc = True)

# Fill in start date where "effective" is missing
ipaws['Effective'] = ipaws['Effective'].fillna(ipaws['Sent'])


In [ ]:
# Identify and remove empty columns & single value columns
ipaws = ipaws.drop([col for col in ipaws.columns if ipaws[col].isnull().all()], axis = 1)
ipaws = ipaws.loc[:, ipaws.nunique() > 1]
ipaws = ipaws.loc[:, ~ipaws.columns.str.contains('Spanish')]

In [ ]:

# Filter to Dates of Interest
ipaws = ipaws[
    (
        (ipaws["Effective"] >= pd.Timestamp('2025-01-07', tz = 'America/Los_Angeles')) & 
        (ipaws["Effective"] <= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles'))
    ) | 
    (
        #pd.isna(ipaws['Effective']) | 
        (ipaws["Expires"] >= pd.Timestamp('2025-01-07', tz = 'America/Los_Angeles')) & 
        (ipaws["Expires"] <= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles'))
    )
]
ipaws = ipaws[ipaws["Sent"] <= pd.Timestamp('2025-01-10', tz = 'America/Los_Angeles')]


We also exclude alerts specifying that a curfew is in effect or that air quality is poor. 

In [ ]:

# Filter to Events of Interest
ipaws = ipaws.loc[~ipaws.Instruction.str.contains('CURFEW', na = True, case = False)] # Curfew
ipaws = ipaws.loc[~ipaws.Category.str.match('Health', na = True, case = False)] # Air Quality




We determine whether an alert was an evacuation order or an evacuation warning based on the "Event" type specified in the data set. However, two alerts were not categorized as "Evacuation Immediate" events, but still explicitly instructed residents to "LEAVE NOW" in the body of the message, so we code these as evacuation orders. 

In [ ]:
# Determine Type of Alert
ipaws['CleanedType'] = np.select(
    [   
        (ipaws['AlertID'] == 153), # Contained messages saying to evacuate despite non-matching Event code
        (ipaws['AlertID'] == 166), # Contained messages saying to evacuate despite non-matching Event code
        (ipaws.Event.str.match('Evacuation Immediate', na = False)),
        (ipaws.Event.str.match('Fire Warning', na = False)),
        (ipaws.Event.str.match('Local Area Emergency', na = False)),
    ],
    [
        'evacuation',
        'evacuation',
        'evacuation',
        'warning',
        'warning'
    ],
    default = None
)


### Ventura County

We manually add a row here for the Ventura County evacuation warning. Per our email with the county, we know that this warning was in effect from 3:48PM to 7:04PM on January 9, which is entirely within our study period. 

In [ ]:
ventura['CleanedType'] = 'warning'
ventura['Effective'] = pd.Timestamp('2025-01-09 15:48', tz = 'America/Los_Angeles')
ventura['Expires'] = pd.Timestamp('2025-01-09 19:04', tz = 'America/Los_Angeles')
ventura['geometry'] = ventura.geometry.to_crs(ipaws.geometry.crs)

ipaws = pd.concat([ipaws, ventura], ignore_index=True)

### Combine

Finally we combine all areas that were issued an evacuation order and all areas that were issued an evacuation warning over the study period. If an area received _both_ warnings and orders, we only included it in the evacuation order area. 

In [ ]:
ipaws_dissolved = ipaws.loc[:,['CleanedType', 'geometry']].dissolve(by = 'CleanedType')
ipaws_dissolved = ipaws_dissolved.reset_index()
ipaws_dissolved.loc[ipaws_dissolved['CleanedType'] == 'warning', 'geometry'] = ipaws_dissolved.loc[ipaws_dissolved['CleanedType'] == 'warning', 'geometry']. \
    difference(ipaws_dissolved.loc[ipaws_dissolved['CleanedType'] == 'evacuation', 'geometry'], align = False)
ipaws_dissolved.explore('CleanedType')

In [ ]:
pd.options.display.max_colwidth = 250
#pd.options.display.max_rows = None
#pd.options.display.max_columns = None

ipaws

In [ ]:

ipaws.to_parquet('../data/clean/evac_clean.parquet')
ipaws_dissolved.to_parquet('../data/clean/evac_clean_aggregated.parquet')

In [ ]:
ipaws_dissolved.explore('CleanedType')

In [ ]:
ipaws_dissolved